# Range join hint
In this notebook, we are exploring the use of the **range join hint** to optimize join operations in Spark SQL. By applying the range join hint, we instruct the query optimizer to use a more efficient join strategy when joining tables based on a range condition (e.g., BETWEEN, inequalities).

The execution plan output demonstrates how the optimizer leverages the range join hint to avoid expensive cross joins or broadcast joins. Instead, it uses a specialized algorithm that quickly matches rows within the specified range, reducing the amount of data shuffled and improving query performance.

Overall, the range join hint helps make the query faster by:
- Minimizing data movement across the cluster.
- Reducing computational overhead for range-based joins.
- Allowing the optimizer to select a join strategy tailored for range predicates.

The key aspect of the range join hint is the _bin size_ that enables the groups creation for the shuffle join strategy.

In [0]:
%sql
DROP TABLE IF EXISTS events;
DROP TABLE IF EXISTS windows;

In [0]:
%sql
-- 10k events uniformly spread over [0, 1_000_000) seconds.
CREATE TABLE events AS
SELECT
  id                                  AS event_id,
  CAST(rand(1) * 1000000 AS BIGINT)   AS ts_sec,
  CAST(rand(2) * 500    AS DECIMAL(10,2)) AS value
FROM range(0, 10000);

-- 2k windows. Each starts somewhere in [0, 1_000_000) and lasts 50–150 seconds.
CREATE TABLE windows AS
SELECT
  id                                              AS window_id,
  CAST(rand(3) * 1000000 AS BIGINT)               AS start_sec,
  CAST(rand(3) * 1000000 + 50 + rand(4) * 100 AS BIGINT) AS end_sec_raw
FROM range(0, 2000);

-- Fix end_sec so it's always >= start_sec + 50.
CREATE OR REPLACE TABLE windows AS
SELECT
  window_id,
  start_sec,
  start_sec + 50 + CAST(rand(5) * 100 AS BIGINT) AS end_sec
FROM windows;

ANALYZE TABLE events  COMPUTE STATISTICS FOR ALL COLUMNS;
ANALYZE TABLE windows COMPUTE STATISTICS FOR ALL COLUMNS;

SELECT 'events'  AS tbl, count(*) FROM events
UNION ALL
SELECT 'windows',       count(*) FROM windows;

tbl,count(*)
events,10000
windows,2000


In [0]:
%sql
-- choose bin size
SELECT
  APPROX_PERCENTILE(
    CAST(end_sec - start_sec AS DOUBLE),
      ARRAY(0.5, 0.9, 0.99, 0.999, 0.9999)
  ) AS percentiles_50_90_99_999_9999
FROM windows;

percentiles_50_90_99_999_9999
"List(98.0, 139.0, 148.0, 149.0, 149.0)"


This is the naive implementation of the join. The hint is missing here and the complexity is O(N × M). Reads it as: for every event, compare against every window row and evaluate the range predicate. With 1 000 000 events and 10 000 windows that's 10 000 000 000  comparisons (10 billion).

In [0]:
%sql
-- no hint
EXPLAIN FORMATTED
SELECT e.event_id, w.window_id
FROM events e
JOIN windows w
  ON e.ts_sec >= w.start_sec
 AND e.ts_sec <  w.end_sec;
-- no hint = nested loop join

plan
"== Physical Plan == AdaptiveSparkPlan (10) +- == Initial Plan == PhotonResultStage (9) +- PhotonColumnarToRow (8) +- PhotonProject (7) +- PhotonBroadcastNestedLoopJoin Inner BuildRight (6) :- PhotonScan parquet workspace.default.events (1) +- PhotonShuffleExchangeSource (5) +- PhotonShuffleMapStage (4) +- PhotonShuffleExchangeSink (3) +- PhotonScan parquet workspace.default.windows (2) (1) PhotonScan parquet workspace.default.events Output [2]: [event_id#23376L, ts_sec#23377L] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/43a5e2d5-8871-44ac-83ef-d3188ce438d8] ReadSchema: struct RequiredDataFilters: [isnotnull(ts_sec#23377L)] (2) PhotonScan parquet workspace.default.windows Output [3]: [window_id#23382L, start_sec#23383L, end_sec#23384L] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/db8b6c01-e380-497c-98a4-1cf4e986c386] ReadSchema: struct RequiredDataFilters: [isnotnull(start_sec#23383L), isnotnull(end_sec#23384L)] (3) PhotonShuffleExchangeSink Input [3]: [window_id#23382L, start_sec#23383L, end_sec#23384L] Arguments: SinglePartition (4) PhotonShuffleMapStage Input [3]: [window_id#23382L, start_sec#23383L, end_sec#23384L] Arguments: EXECUTOR_BROADCAST, [id=#11973] (5) PhotonShuffleExchangeSource Input [3]: [window_id#23382L, start_sec#23383L, end_sec#23384L] (6) PhotonBroadcastNestedLoopJoin Join type: Inner Join condition: ((ts_sec#23377L >= start_sec#23383L) AND (ts_sec#23377L < end_sec#23384L)) (7) PhotonProject Input [5]: [event_id#23376L, ts_sec#23377L, window_id#23382L, start_sec#23383L, end_sec#23384L] Arguments: [event_id#23376L, window_id#23382L] (8) PhotonColumnarToRow Input [2]: [event_id#23376L, window_id#23382L] (9) PhotonResultStage Input [2]: [event_id#23376L, window_id#23382L] (10) AdaptiveSparkPlan Output [2]: [event_id#23376L, window_id#23382L] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = events, windows"


In the next cell you can see range join hint (`/*+ RANGE_JOIN(w, 100) */`) in action.

The execution plan creates _binId_ columns on both sides:
- The events table is not shuffled and Photon creates this additional column during the `SELECT` statement:
```
(2) PhotonProject
Input [2]: [event_id#14580L, ts_sec#14581L]
Arguments: [event_id#14580L, ts_sec#14581L, (ts_sec#14581L div 100) AS pointBinId#14606L]

```

- The smaller windows table also gets this _binId_ column but it's created from a `rangejoinbingenerator`:
```
(4) PhotonGenerate
Input [3]: [window_id#14583L, start_sec#14584L, end_sec#14585L]
Arguments: rangejoinbingenerator((least(start_sec#14584L, end_sec#14585L) div 100), (greatest(start_sec#14584L, end_sec#14585L) div 100), 100.0), [window_id#14583L, start_sec#14584L, end_sec#14585L], false, [binId#14604L, isLastBin#14605]

(5) PhotonProject
Input [5]: [window_id#14583L, start_sec#14584L, end_sec#14585L, binId#14604L, isLastBin#14605]
Arguments: [window_id#14583L, start_sec#14584L, end_sec#14585L, binId#14604L]
```

The window table is later broadcasted...
```
(6) PhotonShuffleExchangeSink
Input [4]: [window_id#14583L, start_sec#14584L, end_sec#14585L, binId#14604L]
Arguments: SinglePartition

(7) PhotonShuffleMapStage
Input [4]: [window_id#14583L, start_sec#14584L, end_sec#14585L, binId#14604L]
Arguments: EXECUTOR_BROADCAST, [id=#7912]

```

...and not matched windows are filtered out with the join condition:
```
(9) PhotonBroadcastHashJoin
Left keys [1]: [pointBinId#14606L]
Right keys [1]: [binId#14604L]
Join type: Inner
Join condition: pointinrange(((ts_sec#14581L >= start_sec#14584L) AND (ts_sec#14581L < end_sec#14585L)), ts_sec#14581L, least(start_sec#14584L, end_sec#14585L), greatest(start_sec#14584L, end_sec#14585L), 100.0)
```

Why it's more optimal? A hash join is not a nested join so the complexity decreases to O(M + N): 
* O(M) is for creating the hash table (binId as the hash key)
* O(N) is for probing the hash table, once per event (events table)


In [0]:
%sql
-- Range join hint
EXPLAIN FORMATTED
SELECT /*+ RANGE_JOIN(w, 100) */
  e.event_id, w.window_id
FROM events e
JOIN windows w
  ON e.ts_sec >= w.start_sec
 AND e.ts_sec <  w.end_sec;

-- Look for pointinrange in the execution plan and rangejoinbingenerator

plan
"== Physical Plan == AdaptiveSparkPlan (13) +- == Initial Plan == PhotonResultStage (12) +- PhotonColumnarToRow (11) +- PhotonProject (10) +- PhotonBroadcastHashJoin Inner (9) :- PhotonProject (2) : +- PhotonScan parquet workspace.default.events (1) +- PhotonShuffleExchangeSource (8) +- PhotonShuffleMapStage (7) +- PhotonShuffleExchangeSink (6) +- PhotonProject (5) +- PhotonGenerate (4) +- PhotonScan parquet workspace.default.windows (3) (1) PhotonScan parquet workspace.default.events Output [2]: [event_id#23418L, ts_sec#23419L] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/43a5e2d5-8871-44ac-83ef-d3188ce438d8] ReadSchema: struct RequiredDataFilters: [isnotnull(ts_sec#23419L)] (2) PhotonProject Input [2]: [event_id#23418L, ts_sec#23419L] Arguments: [event_id#23418L, ts_sec#23419L, (ts_sec#23419L div 100) AS pointBinId#23447L] (3) PhotonScan parquet workspace.default.windows Output [3]: [window_id#23424L, start_sec#23425L, end_sec#23426L] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/db8b6c01-e380-497c-98a4-1cf4e986c386] ReadSchema: struct RequiredDataFilters: [isnotnull(start_sec#23425L), isnotnull(end_sec#23426L)] (4) PhotonGenerate Input [3]: [window_id#23424L, start_sec#23425L, end_sec#23426L] Arguments: rangejoinbingenerator((least(start_sec#23425L, end_sec#23426L) div 100), (greatest(start_sec#23425L, end_sec#23426L) div 100), 100.0), [window_id#23424L, start_sec#23425L, end_sec#23426L], false, [binId#23445L, isLastBin#23446] (5) PhotonProject Input [5]: [window_id#23424L, start_sec#23425L, end_sec#23426L, binId#23445L, isLastBin#23446] Arguments: [window_id#23424L, start_sec#23425L, end_sec#23426L, binId#23445L] (6) PhotonShuffleExchangeSink Input [4]: [window_id#23424L, start_sec#23425L, end_sec#23426L, binId#23445L] Arguments: SinglePartition (7) PhotonShuffleMapStage Input [4]: [window_id#23424L, start_sec#23425L, end_sec#23426L, binId#23445L] Arguments: EXECUTOR_BROADCAST, [id=#12075] (8) PhotonShuffleExchangeSource Input [4]: [window_id#23424L, start_sec#23425L, end_sec#23426L, binId#23445L] (9) PhotonBroadcastHashJoin Left keys [1]: [pointBinId#23447L] Right keys [1]: [binId#23445L] Join type: Inner Join condition: pointinrange(((ts_sec#23419L >= start_sec#23425L) AND (ts_sec#23419L < end_sec#23426L)), ts_sec#23419L, least(start_sec#23425L, end_sec#23426L), greatest(start_sec#23425L, end_sec#23426L), 100.0) (10) PhotonProject Input [7]: [event_id#23418L, ts_sec#23419L, pointBinId#23447L, window_id#23424L, start_sec#23425L, end_sec#23426L, binId#23445L] Arguments: [event_id#23418L, window_id#23424L] (11) PhotonColumnarToRow Input [2]: [event_id#23418L, window_id#23424L] (12) PhotonResultStage Input [2]: [event_id#23418L, window_id#23424L] (13) AdaptiveSparkPlan Output [2]: [event_id#23418L, window_id#23424L] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = events, windows"
